
### Install Duck Duck Go and Trafilatura (scrapping tool) libraries
##### (another scrapping tool is 'Beautiful Soup')

#### also AgentAI is also known as ReAct (Reasoning and acting) Refer the link below
https://www.ibm.com/think/topics/react-agent

### Eval Podcast -  https://www.youtube.com/watch?v=BsWxPI9UM4c
### Anthropic document explaining agents vs workflow - https://www.anthropic.com/engineering/building-effective-agents

In [ ]:
# one time installation; commented after installation
# !pip install ddgs trafilatura

### step1: Setup

In [3]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint
from IPython.display import Markdown, display
from ddgs import DDGS
import trafilatura


load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception ("API key is missing.")
else:
    print(OPENAI_API_KEY[:8])

client = OpenAI()

MODEL = "gpt-4.1-mini"

sk-proj-


### Step 2 : Define the tools

In [4]:
def search_web(query: str):
    """ Search the web using DubkDuckGo browser. Reeturn 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f" \u2705 Got results")
    return json.dumps(results, indent=2) # this step is to covert list into text as LLM underestands only text and not the list

In [5]:

def fetch_url(url:str):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f" \u2705 Got text: {len(text)} chars")
            return text
    print(f" \u274C failed to fech or extract text from url: {url}")
    return f" Could not extract text from {url}. Try a different source"




In [6]:


# test search_eb
message = "AI in healthcare in 2030"
searchRet = search_web(message)
searchRet


 ✅ Got results


'[\n  {\n    "title": "AI in Healthcare 2030: Opportunities, Challenges, and Strategic Insights",\n    "href": "https://www.linkedin.com/pulse/ai-healthcare-2030-opportunities-challenges-strategic-riken-shah-pk86f",\n    "body": "Strategic Insights: Preparing For 2030. By 2030, AI will be an active partner, learning, adapting, and collaborating with clinicians. To leverage its potential: Invest in clinician AI literacy \\u2013 Ensure staff understand AI\\u2019s capabilities, limitations, and ethical use."\n  },\n  {\n    "title": "Artificial Intelligence (AI) In Healthcare Market Growth... | Technavio",\n    "href": "https://www.technavio.com/report/artificial-intelligence-market-in-healthcare-sector-industry-analysis",\n    "body": "Ada Health GmbH - Delivers AI-powered digital health diagnostics, providing advanced symptom assessment and critical patient insights to enhance care delivery.What is the expected growth of the Artificial Intelligence (AI) In Healthcare Market between 2026

In [7]:
# test fetch url
url = "https://www.grandviewresearch.com/industry-analysis/artificial-intelligence-ai-healthcare-market"
fetch_url(url)

 ✅ Got text: 27811 chars


'- Home\n- »\n- Healthcare IT\n- »\n-\nAI In Healthcare Market Size & Share, Industry Report, 2033GVR Report cover\nArtificial Intelligence In Healthcare Market (2026 - 2033) Size, Share & Trends Analysis Report By Component (Hardware, Software, Services), By Application (Clinical Trials, Cybersecurity), By End-use, By Technology, By Region, And Segment Forecasts\n- Report ID: GVR-3-68038-951-7\n- Number of Report Pages: 150\n- Format: PDF\n- Historical Range: 2021 - 2025\n- Forecast Period: 2026 - 2033\n- Industry: Healthcare\n- Report Summary\n- Table of Contents\n- Segmentation\n- Methodology\n- Download FREE Sample\n- Download Sample Report\nAI In Healthcare Market Summary\nThe global artificial intelligence in healthcare market size was estimated at USD 36.67 billion in 2025 and is projected to reach USD 505.59 billion by 2033, growing at a CAGR of 38.90% from 2026 to 2033. The key factor driving market growth is the increasing demand in the healthcare sector for enhanced efficien

In [8]:



search_web_function = {

    "name" : "search_web",
    "description": "Search the web using DubkDuckGo browser. Reeturn 3 results",
    "parameters": {
        "type": "object",
        "properties": {

            "query": {
                "type": "string",
                "description": "The search query to find relevant websites"
            }
        },
        "required":["query"]
    }
}
tools = [{"type": "function", "function": search_web_function}]

fetch_url_function = {

    "name" : "fetch_url",
    "description": "Fetch and extract the main text content from a web page",
    "parameters": {
        "type": "object",
        "properties": {

            "url": {
                "type": "string",
                "description": "The url of the web page to fetch and extract text from"
            }
        },
        "required":["url"]
    }
}

tools.append({"type": "function", "function": fetch_url_function})

tools

[{'type': 'function',
  'function': {'name': 'search_web',
   'description': 'Search the web using DubkDuckGo browser. Reeturn 3 results',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The search query to find relevant websites'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'fetch_url',
   'description': 'Fetch and extract the main text content from a web page',
   'parameters': {'type': 'object',
    'properties': {'url': {'type': 'string',
      'description': 'The url of the web page to fetch and extract text from'}},
    'required': ['url']}}}]

### Step3: Tool Call Hander

In [9]:
# handle tool call
def handle_tool_call(tool_calls):
    # .....
    # return what to add to our "coontext" about the tool call results, a dictionary

    tool_call_results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f" \U0001F527 Calling function {function_name} with argument: {args}")
        if(function_name == "search_web"):
            
            #send notification
            result = search_web(args["query"])
            content = f"Search Web Results: {result}"
            print(content)
            
        elif (function_name == "fetch_url"):
            result = fetch_url(args["url"])
            content = f"Fetched URL content: {result}"
        else:
            content = f"Unknown function: {function_name}"
     
        tool_call_result = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_call.function.name,
                "content": content
            }
        tool_call_results.append(tool_call_result)

    return tool_call_results

### Steep 4: The System Prompt
### This tells the LLM who it is and how to behave. The key things:

### 1) What it job is?
### 2) What tools it has?
### 3) What process to follow?
### 4) what output format to product?

In [ ]:

# kirill vesion 
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word 'DONE:' is a control signal and not a label. Only use the word "DONE:" as per the instruction below -- it has to come at the start of a message

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather informaton from at least 6 distinct sources before delviering your brief.
if you have a fewer than 6 sources then keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that DONE: should be at the start of the final response, so it can be easily extracted and parsed.
You CANNOT and SHOULD NOT include DONE: in any part of the response except at the very beginning of the final response.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

### Step5 : The Agentic Loop

In [11]:
def run_research_agent(topic: str, max_iterations: int = 10) -> str:
    """
        Run the research agent on a topic and return the research brief
        Args:
            topic: The topic to research
            max_iterations: safetly limit to prevent infinite loop
    
    """
    print(f"\n\u0001F50D Starting research on {topic}")

    # Initialize conversation messages list with system prompt + research talk
    messages = [{"role": "system", "content": RESEARCH_AGENT_PROMPT},
                {"role": "user", "content": f"Reserch the following topic and produce a comprehensive research brief:\n {topic}"}]
    #Loop
        #1. Call the LLM and get response
        #2. Chek if LLM called tools

        #3. Otherwise: no tools were called, read message content
            #Check if DONE:, then return
            #Otherwise: not yet done, append message
        #4 if we are entering the final iteration, force a final answer

    #fallback return
    iteration = 1
    while iteration <= max_iterations:
   
        response = client.chat.completions.create(
            model = MODEL,
            messages = messages,
            tools = tools
        )

        message = response.choices[0].message
        messages.append(message)
        print(f"*******iteration: {iteration}\n")
        if message.tool_calls:
            from pprint import pprint
            pprint(message.tool_calls)

            tool_call_results = handle_tool_call(message.tool_calls)
       
            messages.extend(tool_call_results)
        else: # no tool calls, so there just a normal text response from the LLM
            content = message.content
            if content.startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()
                print(f"\n\u2705 Research complete!")
                return research_brief
            else:
                print(f"  \u001F4AD Agent is thinking:")
                pprint(content)
                #loop continues to next iteration
        
        if(iteration == max_iterations -1):
            print("  \u26a0 Safety limit reachd. Stopping research in next iteeration")
            messages.append({"role": "user", "content": "You have reached a maximum number of iterations, Please deliver your response. Please deliver your response with DONE: followed by breif"})
    
        iteration+=1 # increment the count
    return "Research incomplete! Max iterations reached without finalzing the brief"

### Lets Run it

In [12]:
# brief = run_research_agent("AI in healthcare in 2030")
brief = run_research_agent("Prediction of the IPL 2026 winner based on the current status")
display(Markdown(brief))


F50D Starting research on Prediction of the IPL 2026 winner based on the current status
*******iteration: 1

[ChatCompletionMessageFunctionToolCall(id='call_wx4VGiGIhuIsBtL7HbUa5Yiu', function=Function(arguments='{"query":"IPL 2026 winner prediction current status"}', name='search_web'), type='function')]
 🔧 Calling function search_web with argument: {'query': 'IPL 2026 winner prediction current status'}
 ✅ Got results
Search Web Results: [
  {
    "title": "IPL Predictions (2026) - MI Vs SRH Match Winner Prediction",
    "href": "https://iplprediction.com.in/",
    "body": "11 hours ago - IPL 2026 is set to be one of the ... advantage, this IPL Prediction gives Royal Challengers Bengaluru (22% win probability) a slight edge over Mumbai Indians (20%) and Punjab Kings (16%)...."
  },
  {
    "title": "IPL 2026 winner odds & latest betting: Who will win Indian Premier League this season? | Sporting News India",
    "href": "https://www.sportingnews.com/in/betting/news/ipl-2026-winner-o

Comprehensive Research Brief on IPL 2026 Winner Prediction Based on Current Status

Key Facts and Statistics:
- IPL 2026 (IPL 19) runs from March 26 to May 31, 2026, featuring 10 teams playing 84 matches—the largest edition in IPL history.
- The format is a full double round robin, each team playing every other twice, with top four advancing to playoffs.
- Royal Challengers Bengaluru (RCB) are defending champions, having won their maiden title in 2025.
- Current IPL 2026 Points Table (as of late April 2026):
  1. Punjab Kings (PBKS) - 13 points (6W, 1L, 1NR)
  2. Royal Challengers Bengaluru (RCB) - 12 points (6W, 2L)
  3. Rajasthan Royals (RR) - 12 points (6W, 3L)
  4. Sunrisers Hyderabad (SRH) - 10 points (5W, 3L)
  5. Gujarat Titans (GT) - 8 points (4W, 4L)
  6. Chennai Super Kings (CSK) - 6 points (3W, 5L)
  7. Delhi Capitals (DC) - 6 points (3W, 5L)
  8. Kolkata Knight Riders (KKR) - 5 points (2W, 5L, 1NR)
  9. Mumbai Indians (MI) - 4 points (2W, 5L)
  10. Lucknow Super Giants (LSG) - 4 points (2W, 6L)

Main Themes and Arguments:
1. Preseason Predictions and Probabilities:
   - RCB lead with a 22% win probability, favored due to squad continuity from their 2025 title, home advantage at Chinnaswamy Stadium (9 home games), and strong batting lineup led by Virat Kohli.
   - Mumbai Indians are closely behind at 20%, buoyed by experienced stars like Bumrah and Hardik Pandya, and motivated after missing 2025 playoffs.
   - Punjab Kings hold 16% probability, driven by team stability, momentum from 2025 finals, and a settled squad.
   - Other notable contenders include Gujarat Titans (13%) and Delhi Capitals, the latter considered underrated due to strong spin bowling and balanced squad depth.

2. Current Season Performance and Standings:
   - Punjab Kings currently top the points table unbeaten in league losses, showing strong form in IPL 2026.
   - RCB maintains second position, consistent with preseason expectations.
   - Rajasthan Royals and Sunrisers Hyderabad also have strong points tallies, considered wildcard and potential challengers.
   - Mumbai Indians and Lucknow Super Giants surprisingly underperforming relative to past success and expectations.
   - Chennai Super Kings and Delhi Capitals are lagging but still in playoff contention.

3. Team Strengths and Challenges:
   - RCB’s strengths include powerful batting, captain Rajat Patidar's leadership, and strong home record; weakness is slightly thin wrist-spin options on subcontinental slow tracks.
   - Mumbai Indians’ lethal pace attack remains a threat; fitness and middle order depth key uncertainties.
   - Punjab Kings’ settled squad with strong bowling and top order performing well.
   - Rajasthan Royals rely on explosive young batting and aggressive bowling but suffer inconsistency.
   - Delhi Capitals have strong spin bowling options and versatile batting but need to settle batting order early.
   - Teams like KKR and LSG face challenges with squad balance and consistency.

4. Key Player Predictions:
   - Virat Kohli (RCB) is a top Orange Cap contender (most runs), favored by home advantage.
   - Jasprit Bumrah (MI) is favorite for Purple Cap (most wickets), given his unique death bowling skills.
   - Emerging players like Vaibhav Suryavanshi (RR), Cameron Green (KKR), and Kuldeep Yadav (DC) may significantly influence outcomes.
   - Rishabh Pant (LSG) is notable for captaincy and costliest player status, with a pivotal comeback story.

5. Structural and Strategic Factors:
   - The expansion to 84 matches allows greater testing of squad depth and reduces outcome variance.
   - Restoration of home-and-away format benefits teams with strong home venue advantages: RCB at M. Chinnaswamy, MI at Wankhede.
   - Tactical leadership in the captaincy and in-match decisions expected to be decisive.

6. Betting and Fantasy Insights:
   - Betting markets favor RCB and MI; PBKS is a strong outsider.
   - Fantasy strategies emphasize selecting all-rounders and players suited to pitch conditions.
   - Responsible betting is advised given the inherent volatility in T20 cricket.

Summary Conclusion:
The IPL 2026 winner prediction based on current status favors Royal Challengers Bengaluru as the most probable champions due to their defending champion status, squad continuity, and home advantage. Punjab Kings lead the current points table and remain strong contenders along with Mumbai Indians, who possess experienced talent but have had a slow start. Rajasthan Royals and Sunrisers Hyderabad have the potential to disrupt the top four, while Delhi Capitals sit as dark horses. The unpredictability of T20 cricket, player fitness, and form fluctuations keep the tournament outlook dynamic, but statistical models and expert opinions converge on RCB, PBKS, and MI as the top three teams vying for the IPL 2026 title.

Source URLs for Attribution:
1. https://iplprediction.com.in/
2. https://www.sportingnews.com/in/betting/news/ipl-2026-winner-odds-latest-betting-who-will-win-indian-premier-league/73233667b1cd1bc62c51c82d
3. https://iplallmatchprediction.com/
4. https://crictips.com/cricket-predictions/ipl/
5. https://betgully.com/ipl-2026-winner-prediction/
6. https://www.crictracker.com/t20/ipl-indian-premier-league/points-table/?ref=hm

This brief integrates data from preseason analyses, current season standings, expert opinion, betting market insights, and ongoing IPL 2026 form to provide a comprehensive prediction on the likely tournament outcome.

### Evals

In [13]:
# vibe coding your judge prompt

"""
Please write an LLM-as-a-judge prompt for testing the following failure: Research Agent failed to use at least  6 different sources in the deliverable research brief. 

Note: we're not testing correct formal citation, rather we're simply testing the breadth of research by checking if the Agent referenced 6 sources. 

Use the attached example as a reference on how I'd like the prompt structured
"""

# actual judge prompt from vibe coding
# LLM-as-a-Judge Prompt: Source Breadth Validation (Minimum 6 Sources)
JUDGE_PROMPT = """
You are an evaluation system assessing whether a research brief includes at least 6 distinct sources.

Only evaluate source COUNT (not quality or formatting).

A source includes:
- URLs/domains
- Organizations (e.g., WHO, McKinsey)
- Reports/studies/books

Rules:
- Same domain = 1 source
- Same organization repeated = 1 source
- Be conservative

Return STRICT JSON:
{
  "result": "pass" | "fail",
  "source_count": <int>,
  "unique_sources": ["..."],
  "reasoning": "brief explanation"
}

"""

In [21]:
def judge(text: str) -> dict:
    resp = client.responses.create(
        model="gpt-4.1",
        temperature=0,
        input=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": text}
        ]
    )

    raw = resp.output[0].content[0].text.strip()
    return json.loads(raw)


In [15]:
topics = [
    "Impact of generative AI on enterprise SaaS pricing models",
    "Global semiconductor supply chain risks and geopolitics",
    "Climate change effects on coastal real estate markets",
    "Future of fintech regulation in the United States",
    "Advancements in cancer immunotherapy treatments",
    "Electric vehicle adoption trends and infrastructure challenges",
    "Cybersecurity threats in cloud-native architectures",
    "Economic implications of remote work post-pandemic",
    "AI in education: personalized learning outcomes and risks",
    "Space economy growth and commercial satellite markets"
]

In [22]:
import contextlib
import io

# ---- Run Evals _____
results = []
passes = 0

for i, topic in enumerate(topics):
    print(f"[{i+1}/{len(topics)}]  Researching topic: {topic[:60]}...")
    with contextlib.redirect_stdout(io.StringIO()): # suppress printout from the agent for cleaner evaluation logs
        brief = run_research_agent(topic)
    eval_result = judge(brief)

    row = {
        "topic": topic,
        "result": eval_result["result"],
        "source_count": eval_result["source_count"]
    }
    results.append(row)

    if eval_result["result"] == "pass":
        passes += 1

# Summary
summary = {
    "total": len(topics),
    "passed": passes,
    "failed": len(topics) - passes,
    "pass_rate": round(passes / len(topics), 2),
    "results": results
}

print(json.dumps(summary, indent=2))

# Optional: fail fast for CI
assert passes == len(topics), f"{len(topics)-passes} topics failed source requirement"

[1/10]  Researching topic: Impact of generative AI on enterprise SaaS pricing models...
[2/10]  Researching topic: Global semiconductor supply chain risks and geopolitics...
[3/10]  Researching topic: Climate change effects on coastal real estate markets...
[4/10]  Researching topic: Future of fintech regulation in the United States...
[5/10]  Researching topic: Advancements in cancer immunotherapy treatments...
[6/10]  Researching topic: Electric vehicle adoption trends and infrastructure challeng...
[7/10]  Researching topic: Cybersecurity threats in cloud-native architectures...
[8/10]  Researching topic: Economic implications of remote work post-pandemic...
[9/10]  Researching topic: AI in education: personalized learning outcomes and risks...
[10/10]  Researching topic: Space economy growth and commercial satellite markets...
{
  "total": 10,
  "passed": 10,
  "failed": 0,
  "pass_rate": 1.0,
  "results": [
    {
      "topic": "Impact of generative AI on enterprise SaaS pricing m

### Eval 2 (forcing to fail by lossening the prompt)

In [23]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word 'DONE:' is a control signal and not a label. Only use the word "DONE:" as per the instruction below -- it has to come at the start of a message

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. Try to get information from at least 6 different sources.
7. When you have enough information  synthesize into a research brief


When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that DONE: should be at the start of the final response, so it can be easily extracted and parsed.
You CANNOT and SHOULD NOT include DONE: in any part of the response except at the very beginning of the final response.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

In [27]:
import contextlib
import io

# ---- Run Evals _____
results = []
passes = 0

for i, topic in enumerate(topics):
    print(f"\n\n-----------------------------\n\n[{i+1}/{len(topics)}]  Researching topic: {topic[:60]}...")
    with contextlib.redirect_stdout(io.StringIO()): # suppress printout from the agent for cleaner evaluation logs
        brief = run_research_agent(topic)
    print(brief)
    eval_result = judge(brief)

    row = {
        "topic": topic,
        "result": eval_result["result"],
        "source_count": eval_result["source_count"]
    }
    results.append(row)

    if eval_result["result"] == "pass":
        passes += 1

# Summary
summary = {
    "total": len(topics),
    "passed": passes,
    "failed": len(topics) - passes,
    "pass_rate": round(passes / len(topics), 2),
    "results": results
}

print(json.dumps(summary, indent=2))

# Optional: fail fast for CI
# assert passes == len(topics), f"{len(topics)-passes} topics failed source requirement"



-----------------------------

[1/10]  Researching topic: Impact of generative AI on enterprise SaaS pricing models...
Research Brief: Impact of Generative AI on Enterprise SaaS Pricing Models

Key Facts and Statistics:
- Enterprise generative AI spending reached $37 billion in 2025, a 3.2x increase from 2024, with $19 billion going to applications and $18 billion to infrastructure. (Source: MenloVentures report)
- The global AI SaaS market is forecasted to grow from $20 billion in 2025 to nearly $86 billion by 2032. (Source: Innovecs)
- Over 42% of SaaS industry leaders see generative AI as transformative, especially impacting pricing, sales, and product development. (Source: L.E.K. Consulting)
- Microsoft Copilot-style AI features command a premium of 60%-70% over base SaaS subscription fees. (Source: L.E.K. Consulting)
- SaaS providers that integrate AI report productivity gains around 50%, with task automation leading to significant operational cost savings.

Main Themes and Argu